In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# --- STEP 1: LOADING THE DATASET ---
# Loading feature names from the metadata file and cleaning possible whitespaces
features = pd.read_csv("UNSW-NB15_features.csv", encoding='ISO-8859-1')
feature_names = features['Name'].str.strip().tolist()

# Iteratively loading the 4 CSV files provided in the UNSW-NB15 raw dataset
dfs = []
for i in range(1, 5):
    df = pd.read_csv(f"UNSW-NB15_{i}.csv", header=None, low_memory=False)
    df.columns = feature_names
    dfs.append(df)

# Merging all data parts into a single master DataFrame for holistic pre-processing
df_all = pd.concat(dfs, ignore_index=True, sort=False)
df_all.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


In [3]:
# --- STEP 2: FEATURE SELECTION ---
# Dropping IP addresses, Port numbers, and Timestamps as they represent session-specific
# metadata that can lead to overfitting. 'attack_cat' is removed to avoid data leakage
# since our objective is binary 'Label' classification.
drop_cols = ['srcip', 'dstip', 'sport', 'dsport', 'Stime', 'Ltime', 'attack_cat']
df_cleaned = df_all.drop(columns=drop_cols)

In [4]:
# --- STEP 3: HANDLING MISSING VALUES (DOMAIN-SPECIFIC IMPUTATION) ---
# Checking for null values before imputation
null_counts = df_cleaned.isnull().sum()
print(null_counts[null_counts > 0])

# Missing values in 'ct_ftp_cmd' indicate the absence of FTP protocol in the flow.
# Filling with 0 allows the model to process 'lack of activity' as a numerical feature.
df_cleaned['ct_ftp_cmd'] = pd.to_numeric(df_cleaned['ct_ftp_cmd'], errors='coerce').fillna(0).astype(int)

# NaN values in HTTP methods occur in non-HTTP traffic.
# Assigning 0 ensures the model focuses application layer analysis only on relevant protocols.
df_cleaned['ct_flw_http_mthd'] = df_cleaned['ct_flw_http_mthd'].fillna(0)

# NaN values in 'is_ftp_login' represent that no login attempt occurred.
# Standardizing these as 0 provides the model with clear statistical evidence of 'no successful login'.
df_cleaned['is_ftp_login'] = df_cleaned['is_ftp_login'].fillna(0)

# Verifying that all null values have been addressed
print(f"Total null values after imputation: {df_cleaned.isnull().sum().sum()}")

ct_flw_http_mthd    1348145
is_ftp_login        1429879
dtype: int64
Total null values after imputation: 0


In [5]:
# --- STEP 4: CATEGORICAL ENCODING (ONE-HOT ENCODING) ---
df_cleaned.info()
categorical_cols = ['proto', 'service', 'state']

# Using OneHotEncoder with sparse_output=True to optimize memory efficiency for 2.5M+ rows.
# 'drop=first' is used to avoid the dummy variable trap (multicollinearity).
ohe = OneHotEncoder(drop='first', sparse_output=True)
encoded_sparse = ohe.fit_transform(df_cleaned[categorical_cols])

# Converting the sparse matrix to a Sparse DataFrame structure to keep the memory footprint low.
encoded_df = pd.DataFrame.sparse.from_spmatrix(
    encoded_sparse,
    columns=ohe.get_feature_names_out(categorical_cols),
    index=df_cleaned.index
)

# Integrating encoded features back into the main dataset and removing original string columns.
cleaned_dataset = pd.concat([df_cleaned.drop(columns=categorical_cols), encoded_df], axis=1)

print("Number of new columns after encoding:",  cleaned_dataset.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 42 columns):
 #   Column            Dtype  
---  ------            -----  
 0   proto             object 
 1   state             object 
 2   dur               float64
 3   sbytes            int64  
 4   dbytes            int64  
 5   sttl              int64  
 6   dttl              int64  
 7   sloss             int64  
 8   dloss             int64  
 9   service           object 
 10  Sload             float64
 11  Dload             float64
 12  Spkts             int64  
 13  Dpkts             int64  
 14  swin              int64  
 15  dwin              int64  
 16  stcpb             int64  
 17  dtcpb             int64  
 18  smeansz           int64  
 19  dmeansz           int64  
 20  trans_depth       int64  
 21  res_bdy_len       int64  
 22  Sjit              float64
 23  Djit              float64
 24  Sintpkt           float64
 25  Dintpkt           float64
 26  tcprtt        

In [6]:
# --- STEP 5: DATA STANDARDIZATION (SCALING) ---
# The dataset contains extreme value ranges (e.g., 0.001 to 1,500,000).
# We apply standardization (Mean=0, StdDev=1) so the model treats all numerical features with equal weight.
numeric_cols = df_cleaned.select_dtypes(include=['float64', 'int64', 'int32']).columns.tolist()

# Ensuring the target variable 'Label' is excluded from scaling to preserve its binary nature.
if 'Label' in numeric_cols:
    numeric_cols.remove('Label')

scaler = StandardScaler()
cleaned_dataset[numeric_cols] = scaler.fit_transform(cleaned_dataset[numeric_cols])

# Displaying first few rows of scaled numerical columns for verification
print(cleaned_dataset[numeric_cols].head())

        dur    sbytes    dbytes      sttl      dttl     sloss     dloss  \
0 -0.047234 -0.074595 -0.225105 -0.425902 -0.041232 -0.229334 -0.288533   
1 -0.044715 -0.067574 -0.224236 -0.425902 -0.041232 -0.229334 -0.288533   
2 -0.047230 -0.074347 -0.225019 -0.425902 -0.041232 -0.229334 -0.288533   
3 -0.047223 -0.074595 -0.225105 -0.425902 -0.041232 -0.229334 -0.288533   
4 -0.047226 -0.074347 -0.225019 -0.425902 -0.041232 -0.229334 -0.288533   

      Sload     Dload     Spkts  ...  ct_flw_http_mthd  is_ftp_login  \
0 -0.307375 -0.432928 -0.410163  ...         -0.197833     -0.130014   
1 -0.310855 -0.568156 -0.383945  ...         -0.197833     -0.130014   
2 -0.307194 -0.429500 -0.410163  ...         -0.197833     -0.130014   
3 -0.307912 -0.451675 -0.410163  ...         -0.197833     -0.130014   
4 -0.307382 -0.435942 -0.410163  ...         -0.197833     -0.130014   

   ct_ftp_cmd  ct_srv_src  ct_srv_dst  ct_dst_ltm  ct_src_ ltm  \
0   -0.111508   -0.572772   -0.183780   -0.666391 